## Problem 1: Tensors and broadcasting

In [8]:
import torch

t = torch.tensor([
    [1., 2., 3.],
    [4., 5., 6.]
])

bias = torch.tensor([10., 20., 30.])

print("shape:", tuple(t.shape))
print("dtype:", t.dtype)
print("broadcast:", (t + bias).tolist())

shape: (2, 3)
dtype: torch.float32
broadcast: [[11.0, 22.0, 33.0], [14.0, 25.0, 36.0]]


## Problem 2: Autograd matches the by-hand gradient

In [9]:
x = torch.tensor([1., 2., 3.], requires_grad=True)

y = (x ** 2).sum()

y.backward()

print("x.grad:", x.grad)

x.grad: tensor([2., 4., 6.])


## Problem 3: Count the parameters of an nn.Module

In [10]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(64, 16),
    nn.ReLU(),
    nn.Linear(16, 10)
)

total_parameters = 0

for parameter in model.parameters():
    total_parameters += parameter.numel()

print(f"total parameters: {total_parameters}")

total parameters: 1210


## Problem 4: The five-line loop lowers the loss

In [11]:
x = torch.tensor([1., 2., 3., 4.])
y = torch.tensor([2., 4., 6., 8.])

w = torch.tensor([0.0], requires_grad=True)

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD([w], lr=0.02)

first_loss = None

for step in range(80):
    pred = w * x
    loss = loss_fn(pred, y)

    if step == 0:
        first_loss = loss.item()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

last_loss = loss_fn(w * x, y).item()

print(f"first loss: {first_loss:.1f}")
print(f"last loss: {last_loss:.1f}")
print(f"decreased: {last_loss < first_loss}")

first loss: 30.0
last loss: 0.0
decreased: True


## Problem 5: The zero_grad trap

In [12]:
w = torch.tensor([1.0], requires_grad=True)

without_zero_grad = []

for i in range(3):
    loss = (w ** 2).sum()
    loss.backward()
    without_zero_grad.append(w.grad.item())

print("without zero_grad:", without_zero_grad)


w = torch.tensor([1.0], requires_grad=True)

with_zero_grad = []

for i in range(3):
    if w.grad is not None:
        w.grad.zero_()

    loss = (w ** 2).sum()
    loss.backward()

    with_zero_grad.append(w.grad.item())

print("with zero_grad:", with_zero_grad)

without zero_grad: [2.0, 4.0, 6.0]
with zero_grad: [2.0, 2.0, 2.0]


## Problem 6: Logits, not probabilities

In [13]:
print(
    "logits: CrossEntropyLoss applies softmax internally, so pass raw logits; "
    "softmaxing first applies it twice, weakening the signal and hurting training"
)

logits: CrossEntropyLoss applies softmax internally, so pass raw logits; softmaxing first applies it twice, weakening the signal and hurting training
